# SupplyForge AI 2.0: KAN-34 - Historical Lag & Rolling Statistics



Tài liệu này thực hiện việc tính toán các đặc trưng chuỗi thời gian (time-series features) nhằm cung cấp thông tin lịch sử bán hàng cho mô hình học máy LightGBM.



Các đặc trưng bao gồm các chỉ số trễ (lags), thống kê trượt (rolling mean/std/median/sum/zero_sales_ratio), trung bình động lũy thừa (EMA), và số ngày kể từ lần bán cuối cùng (days_since_last_sale). Quy trình được thiết kế độc lập cho từng SKU và triệt tiêu hoàn toàn rò rỉ dữ liệu (Target Leakage) bằng cách lùi cửa sổ tính toán (shift(1)).


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Bảo toàn số lượng dòng**: Kích thước lưới chuỗi thời gian không thay đổi (giữ nguyên 323,250 dòng).

2. **Chống rò rỉ dữ liệu**: Chỉ sử dụng lịch sử lùi ngày (`shift(1)`) làm đầu vào cho mọi thống kê trượt.

3. **Tách biệt SKU**: Sử dụng phân nhóm độc lập theo `itemId`.

4. **Giai đoạn đầu của SKU**: Sử dụng cấu hình `min_periods=1` để xử lý các ngày đầu tiên của SKU mới xuất hiện.


In [1]:
import pandas as pd

import numpy as np

import os



PROJECT_ROOT = '/home/nguyen-dai-tung/supplyforge-ai'

GOLD_DIR = os.path.join(PROJECT_ROOT, 'data', 'gold')

grid_path = os.path.join(GOLD_DIR, 'calendar_daily_sales_metadata.parquet')

if not os.path.exists(grid_path):

    grid_path = os.path.join(GOLD_DIR, 'calendar_daily_sales_metadata.csv')



print("Đang tải lưới dữ liệu từ Gold Layer...")

df_grid = pd.read_parquet(grid_path) if grid_path.endswith(".parquet") else pd.read_csv(grid_path)

print(f"Tổng số dòng: {len(df_grid):,}") 


Đang tải lưới dữ liệu từ Gold Layer...
Tổng số dòng: 1,597,060


In [2]:
def generate_time_series_features(df):
    df_out = df.sort_values(["itemId", "date"]).copy()
    for lag in [1, 7, 14, 28]:
        df_out[f"lag_{lag}"] = df_out.groupby("itemId")["quantity"].shift(lag).astype("float32")
    for window in [7, 14, 28]:
        df_out[f"rolling_sum_{window}"] = df_out.groupby("itemId")["quantity"].transform(lambda x: x.shift(1).rolling(window, min_periods=1).sum()).astype("float32")
        df_out[f"rolling_mean_{window}"] = df_out.groupby("itemId")["quantity"].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean()).astype("float32")
        df_out[f"rolling_std_{window}"] = df_out.groupby("itemId")["quantity"].transform(lambda x: x.shift(1).rolling(window, min_periods=1).std()).fillna(0).astype("float32")
        df_out[f"zero_sales_ratio_{window}"] = df_out.groupby("itemId")["quantity"].transform(lambda x: (x.shift(1) == 0).rolling(window, min_periods=1).mean()).astype("float32")
    df_out["ema_7"] = df_out.groupby("itemId")["quantity"].transform(lambda x: x.shift(1).ewm(span=7, adjust=False).mean()).astype("float32")
    def calc_days_since_last_sale(s):
        mask = s > 0
        idx = np.where(mask, np.arange(len(s)), np.nan)
        ffill_idx = pd.Series(idx).ffill()
        days = np.arange(len(s)) - ffill_idx.shift(1)
        return days.fillna(999).astype("float32")
    df_out["days_since_last_sale"] = df_out.groupby("itemId")["quantity"].transform(calc_days_since_last_sale)
    return df_out

print("Đang chạy quy trình tính toán đặc trưng chuỗi thời gian...")
df_featured = generate_time_series_features(df_grid)
print(f"Kích thước lưới sau khi tính toán: {df_featured.shape}")
print("Danh sách cột đặc trưng mới sinh ra:")
print(list(df_featured.columns))


Đang chạy quy trình tính toán đặc trưng chuỗi thời gian...


Kích thước lưới sau khi tính toán: (1597060, 29)
Danh sách cột đặc trưng mới sinh ra:
['date', 'itemId', 'quantity', 'brand', 'categoryId', 'price', 'cost', 'warehouse', 'brand_encoded', 'categoryId_encoded', 'warehouse_encoded', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_sum_7', 'rolling_mean_7', 'rolling_std_7', 'zero_sales_ratio_7', 'rolling_sum_14', 'rolling_mean_14', 'rolling_std_14', 'zero_sales_ratio_14', 'rolling_sum_28', 'rolling_mean_28', 'rolling_std_28', 'zero_sales_ratio_28', 'ema_7', 'days_since_last_sale']


In [3]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")







# 1. Xác thực số lượng dòng được bảo toàn



print(f"Số dòng trước: {len(df_grid)} | Số dòng sau: {len(df_featured)}") 


assert len(df_grid) == len(df_featured), "Lỗi: Số lượng dòng lưới bị thay đổi!"



print("1. Đảm bảo bảo toàn số dòng lưới: Đạt")







# 2. Xác thực sắp xếp thời gian chronologically theo từng SKU



is_sorted = df_featured.groupby('itemId')['date'].apply(lambda x: x.is_monotonic_increasing).all()



assert is_sorted, "Lỗi: Chuỗi thời gian chưa được sắp xếp tăng dần!"



print("2. Dữ liệu được sắp xếp chronologically cho mỗi SKU: Đạt")







# 3. Xác thực bản ghi đầu tiên của mỗi SKU có lag_1 là NaN



first_records = df_featured.groupby('itemId')['lag_1'].nth(0)



print(f"Số lượng SKU có lag_1 là NaN ở ngày đầu tiên: {first_records.isnull().sum()} / {len(first_records)}") 


assert first_records.isnull().all(), "Lỗi: Có SKU không chứa NaN ở ngày đầu tiên!"



print("3. Lịch sử bắt đầu của mỗi SKU được khởi tạo bằng NaN (đúng logic): Đạt")



=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

Số dòng trước: 1597060 | Số dòng sau: 1597060
1. Đảm bảo bảo toàn số dòng lưới: Đạt
2. Dữ liệu được sắp xếp chronologically cho mỗi SKU: Đạt
Số lượng SKU có lag_1 là NaN ở ngày đầu tiên: 940 / 940
3. Lịch sử bắt đầu của mỗi SKU được khởi tạo bằng NaN (đúng logic): Đạt


In [4]:
print("=== KIỂM CHỨNG TOÁN HỌC ĐỐI CHIẾU THỦ CÔNG TRÊN 20 SKU NGẪU NHIÊN ===\n")







np.random.seed(42)



random_items = np.random.choice(df_featured['itemId'].unique(), size=20, replace=False)







for item in random_items:



    df_item = df_featured[df_featured['itemId'] == item].sort_values('date').reset_index(drop=True)



    



    # Thực hiện đối chiếu thủ công trên dòng thứ 42 (đảm bảo đã tích lũy đủ lịch sử)



    test_idx = 42



    if len(df_item) > test_idx:



        qty_history = df_item['quantity'].iloc[:test_idx].values



        



        # Tính toán thủ công giá trị lùi



        expected_lag1 = qty_history[-1]



        expected_sum7 = sum(qty_history[-7:])



        expected_mean7 = expected_sum7 / 7.0



        



        actual_lag1 = df_item['lag_1'].iloc[test_idx]



        actual_sum7 = df_item['rolling_sum_7'].iloc[test_idx]



        actual_mean7 = df_item['rolling_mean_7'].iloc[test_idx]



        



        # So sánh với độ dung sai số thực



        assert abs(actual_lag1 - expected_lag1) < 1e-5, f"Lỗi đối chiếu lag_1 trên SKU {item}"



        assert abs(actual_sum7 - expected_sum7) < 1e-5, f"Lỗi đối chiếu rolling_sum_7 trên SKU {item}"



        assert abs(actual_mean7 - expected_mean7) < 1e-5, f"Lỗi đối chiếu rolling_mean_7 trên SKU {item}"



        



print("4. Đối chiếu thủ công 20 SKU ngẫu nhiên khớp hoàn toàn 100%: Đạt")







print("\n5. Xem trước mẫu đặc trưng của một SKU cụ thể (10 dòng đầu):")



sample_item = df_featured['itemId'].iloc[0]



df_sample = df_featured[df_featured['itemId'] == sample_item].head(10)



print(df_sample[['date', 'quantity', 'lag_1', 'rolling_sum_7', 'rolling_mean_7', 'days_since_last_sale', 'zero_sales_ratio_7', 'ema_7']].to_string())



=== KIỂM CHỨNG TOÁN HỌC ĐỐI CHIẾU THỦ CÔNG TRÊN 20 SKU NGẪU NHIÊN ===



4. Đối chiếu thủ công 20 SKU ngẫu nhiên khớp hoàn toàn 100%: Đạt

5. Xem trước mẫu đặc trưng của một SKU cụ thể (10 dòng đầu):
            date  quantity  lag_1  rolling_sum_7  rolling_mean_7  days_since_last_sale  zero_sales_ratio_7  ema_7
27    2021-04-22         0    NaN            NaN             NaN                 999.0            0.000000    NaN
967   2021-04-23         0    0.0            0.0             0.0                  17.0            0.500000    0.0
1907  2021-04-24         0    0.0            0.0             0.0                   NaN            0.666667    0.0
2847  2021-04-25         0    0.0            0.0             0.0                   NaN            0.750000    0.0
3787  2021-04-26         0    0.0            0.0             0.0                   NaN            0.800000    0.0
4727  2021-04-27         0    0.0            0.0             0.0                   NaN            0.833333    0.0
5667  2021-04-28         0    0.0            0.0             0.0           

In [5]:
output_parquet = os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.parquet")
output_csv = os.path.join(GOLD_DIR, "calendar_daily_sales_metadata.csv")
df_featured.to_parquet(output_parquet, index=False)
df_featured.to_csv(output_csv, index=False)
print(f"Đã lưu đặc trưng chuỗi thời gian tại Gold Layer: {output_parquet}")


Đã lưu đặc trưng chuỗi thời gian tại Gold Layer: /home/nguyen-dai-tung/supplyforge-ai/data/gold/calendar_daily_sales_metadata.parquet


In [6]:
import pandas as pd

df = pd.read_csv('/home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/calendar_daily_sales_metadata.csv')



sku = "2963443223"



check = (

    df[df.itemId==sku]

    [["date","quantity","lag_1"]]

    .head(10)

)



print(check)


Empty DataFrame
Columns: [date, quantity, lag_1]
Index: []
